In [6]:
import os
import pandas as pd

path = "/content/data_labeled.csv"

if not os.path.exists(path):
    print("ERROR: File not found!")
else:
    file_size_mb = os.path.getsize(path) / (1024 * 1024)

    print("File size (MB):", round(file_size_mb, 2))

    try:
        df_check = pd.read_csv(path)

        print("Dataset shape:", df_check.shape)
        print("Columns:", df_check.columns.tolist())
        print("First timestamp:", df_check["timestamp"].iloc[0])
        print("Last timestamp:", df_check["timestamp"].iloc[-1])
        print("Segment column exists:", "segment" in df_check.columns)
        print("Ground truth column exists:", "ground_truth" in df_check.columns)

    except Exception as error:
        print("ERROR while reading file:", error)

File size (MB): 204.37
Dataset shape: (1516948, 18)
Columns: ['timestamp', 'TP2', 'TP3', 'H1', 'DV_pressure', 'Reservoirs', 'Oil_temperature', 'Motor_current', 'COMP', 'DV_eletric', 'Towers', 'MPG', 'LPS', 'Pressure_switch', 'Oil_level', 'Caudal_impulses', 'segment', 'ground_truth']
First timestamp: 2020-02-01 00:00:00
Last timestamp: 2020-09-01 03:59:50
Segment column exists: True
Ground truth column exists: True


In [7]:
import pandas as pd

path = "/content/data_labeled.csv"

df = pd.read_csv(path, usecols=["timestamp", "segment"])

df["timestamp"] = pd.to_datetime(df["timestamp"])
df = df.sort_values("timestamp")

time_diff = df["timestamp"].diff()

print("Rows:", len(df))
print("Timestamps sorted:", df["timestamp"].is_monotonic_increasing)
print("Most common time difference:")
print(time_diff.value_counts().head(5))

print("\nLargest time gaps:")
print(time_diff.sort_values(ascending=False).head(10))

Rows: 1516948
Timestamps sorted: True
Most common time difference:
timestamp
0 days 00:00:10    1337521
0 days 00:00:09     128277
0 days 00:00:12      38321
0 days 00:00:13       7988
0 days 00:00:11       4471
Name: count, dtype: int64

Largest time gaps:
618538    2 days 00:01:58
1055664   1 days 12:14:36
214850    1 days 04:03:01
1318888   1 days 00:40:33
805415    1 days 00:34:51
1126187   0 days 23:56:00
1454438   0 days 23:39:17
708881    0 days 22:17:41
908125    0 days 21:28:25
450114    0 days 20:43:37
Name: timestamp, dtype: timedelta64[ns]


In [8]:
import pandas as pd

path = "/content/data_labeled.csv"

df = pd.read_csv(
    path,
    usecols=["timestamp", "segment"]
)

df["timestamp"] = pd.to_datetime(df["timestamp"])

segment_summary = (
    df.groupby("segment", dropna=False)
    .agg(
        n_rows=("timestamp", "size"),
        start_time=("timestamp", "min"),
        end_time=("timestamp", "max")
    )
    .reset_index()
)

print("Number of segments:", len(segment_summary))
display(segment_summary)

Number of segments: 332


,segment,n_rows,start_time,end_time
0,0,4654,2020-02-01 00:00:00,2020-02-01 12:48:40
1,1,2460,2020-02-01 12:53:47,2020-02-01 19:40:04
2,2,30,2020-02-01 23:15:33,2020-02-01 23:20:21
3,3,1110,2020-02-02 04:34:27,2020-02-02 07:37:40
4,4,23550,2020-02-02 07:41:52,2020-02-05 00:32:21
...,...,...,...,...
327,327,7290,2020-08-30 04:26:10,2020-08-31 00:30:23
328,328,1740,2020-08-31 01:49:32,2020-08-31 06:36:50
329,329,4920,2020-08-31 06:42:31,2020-08-31 20:15:10
330,330,30,2020-09-01 00:37:33,2020-09-01 00:42:20


In [9]:
import pandas as pd

df = pd.read_csv(
    "/content/data_labeled.csv",
    usecols=["ground_truth"]
)

print(df["ground_truth"].value_counts(dropna=False))

ground_truth
0    1513533
1       3415
Name: count, dtype: int64


In [10]:
import pandas as pd
import os

file_path = "/content/data_labeled.csv"

print("File exists:", os.path.exists(file_path))

df = pd.read_csv(file_path)

print("Dataset shape:", df.shape)
print("Columns:")
print(df.columns.tolist())

File exists: True
Dataset shape: (1516948, 18)
Columns:
['timestamp', 'TP2', 'TP3', 'H1', 'DV_pressure', 'Reservoirs', 'Oil_temperature', 'Motor_current', 'COMP', 'DV_eletric', 'Towers', 'MPG', 'LPS', 'Pressure_switch', 'Oil_level', 'Caudal_impulses', 'segment', 'ground_truth']


In [11]:
segment_summary = df.groupby("segment").agg(
    total_rows=("ground_truth", "size"),
    abnormal_rows=("ground_truth", "sum"),
    start_time=("timestamp", "min"),
    end_time=("timestamp", "max")
)

print("Number of segments:", len(segment_summary))
print(segment_summary.head(10))

Number of segments: 332
         total_rows  abnormal_rows           start_time             end_time
segment                                                                     
0              4654              0  2020-02-01 00:00:00  2020-02-01 12:48:40
1              2460              0  2020-02-01 12:53:47  2020-02-01 19:40:04
2                30              0  2020-02-01 23:15:33  2020-02-01 23:20:21
3              1110              0  2020-02-02 04:34:27  2020-02-02 07:37:40
4             23550              0  2020-02-02 07:41:52  2020-02-05 00:32:21
5               150              0  2020-02-05 00:43:00  2020-02-05 01:07:37
6              7860              0  2020-02-05 04:28:42  2020-02-06 02:07:01
7              2280              0  2020-02-06 07:52:21  2020-02-06 14:08:53
8              1800              0  2020-02-06 14:19:23  2020-02-06 19:16:35
9                90              0  2020-02-06 23:33:19  2020-02-06 23:48:02


In [12]:
abnormal_segments = segment_summary[
    segment_summary["abnormal_rows"] > 0
].sort_values("abnormal_rows", ascending=False)

print("Segments containing abnormal readings:",
      len(abnormal_segments))

print(abnormal_segments.head(20))

print("\nTotal abnormal readings:",
      abnormal_segments["abnormal_rows"].sum())

Segments containing abnormal readings: 3
         total_rows  abnormal_rows           start_time             end_time
segment                                                                     
124            6480           2476  2020-04-27 01:42:01  2020-04-27 19:32:23
102            7379            909  2020-04-12 04:44:35  2020-04-13 01:03:33
123              30             30  2020-04-27 01:12:49  2020-04-27 01:17:36

Total abnormal readings: 3415


In [13]:
for segment_id in [102, 123, 124]:
    segment_data = df[df["segment"] == segment_id].copy()

    print(f"\nSegment {segment_id}")
    print(segment_data["ground_truth"].value_counts().sort_index())

    print("\nFirst and last abnormal timestamps:")
    abnormal = segment_data[segment_data["ground_truth"] == 1]

    if not abnormal.empty:
        print("Start:", abnormal["timestamp"].min())
        print("End:", abnormal["timestamp"].max())


Segment 102
ground_truth
0    6470
1     909
Name: count, dtype: int64

First and last abnormal timestamps:
Start: 2020-04-12 10:00:00
End: 2020-04-12 12:30:00

Segment 123
ground_truth
1    30
Name: count, dtype: int64

First and last abnormal timestamps:
Start: 2020-04-27 01:12:49
End: 2020-04-27 01:17:36

Segment 124
ground_truth
0    4004
1    2476
Name: count, dtype: int64

First and last abnormal timestamps:
Start: 2020-04-27 01:42:01
End: 2020-04-27 08:30:55


In [14]:
for segment_id in [102, 123, 124]:
    data = df[df["segment"] == segment_id].copy()
    data["timestamp"] = pd.to_datetime(data["timestamp"])
    data = data.sort_values("timestamp")

    data["label_change"] = (
        data["ground_truth"] != data["ground_truth"].shift()
    )

    print(f"\nSegment {segment_id}")
    print("Label transitions:", data["label_change"].sum())

    print("Label sequence:")
    print(data["ground_truth"].value_counts().sort_index())


Segment 102
Label transitions: 3
Label sequence:
ground_truth
0    6470
1     909
Name: count, dtype: int64

Segment 123
Label transitions: 1
Label sequence:
ground_truth
1    30
Name: count, dtype: int64

Segment 124
Label transitions: 2
Label sequence:
ground_truth
0    4004
1    2476
Name: count, dtype: int64


In [15]:
for segment_id in [102, 123, 124]:
    data = df[df["segment"] == segment_id].copy()
    data["timestamp"] = pd.to_datetime(data["timestamp"])
    data = data.sort_values("timestamp").reset_index(drop=True)

    changes = data[
        data["ground_truth"] != data["ground_truth"].shift()
    ].copy()

    changes = changes.iloc[1:]

    print(f"\nSegment {segment_id}")
    print(changes[["timestamp", "ground_truth"]].to_string(index=False))


Segment 102
          timestamp  ground_truth
2020-04-12 10:00:00             1
2020-04-12 12:30:10             0

Segment 123
Empty DataFrame
Columns: [timestamp, ground_truth]
Index: []

Segment 124
          timestamp  ground_truth
2020-04-27 08:31:04             0


In [16]:
for segment_id in [102, 123, 124]:
    data = df[df["segment"] == segment_id].copy()
    data["timestamp"] = pd.to_datetime(data["timestamp"])
    data = data.sort_values("timestamp")

    intervals = data["timestamp"].diff().dt.total_seconds()

    print(f"\nSegment {segment_id}")
    print("Median interval (seconds):", intervals.median())
    print("Minimum interval (seconds):", intervals.min())
    print("Maximum interval (seconds):", intervals.max())


Segment 102
Median interval (seconds): 10.0
Minimum interval (seconds): 9.0
Maximum interval (seconds): 10.0

Segment 123
Median interval (seconds): 10.0
Minimum interval (seconds): 9.0
Maximum interval (seconds): 10.0

Segment 124
Median interval (seconds): 10.0
Minimum interval (seconds): 9.0
Maximum interval (seconds): 10.0


In [17]:
import pandas as pd

df["timestamp"] = pd.to_datetime(df["timestamp"])

data = df.sort_values(
    ["segment", "timestamp"]
).copy()

data["time_diff"] = (
    data.groupby("segment")["timestamp"]
    .diff()
    .dt.total_seconds()
)

print("Overall Interval Statistics:")
print(data["time_diff"].describe())

print("\nInterval Frequency:")
print(
    data["time_diff"]
    .value_counts()
    .sort_index()
    .head(20)
)

print("\nNumber of Missing Intervals:")
print(data["time_diff"].isna().sum())

print("\nNumber of Intervals Greater Than 10 Seconds:")
print((data["time_diff"] > 10).sum())

print("\nNumber of Non-Positive Intervals:")
print((data["time_diff"] <= 0).sum())

print("\nTotal Rows:", len(data))
print("Total Segments:", data["segment"].nunique())

Overall Interval Statistics:
count    1.516616e+06
mean     9.984945e+00
std      4.884339e-01
min      8.000000e+00
25%      1.000000e+01
50%      1.000000e+01
75%      1.000000e+01
max      3.000000e+01
Name: time_diff, dtype: float64

Interval Frequency:
time_diff
8.0           2
9.0      128277
10.0    1337521
11.0       4471
12.0      38321
13.0       7988
14.0          3
15.0          1
17.0          3
18.0          1
19.0          5
20.0          3
21.0         10
22.0          4
23.0          3
26.0          1
27.0          1
30.0          1
Name: count, dtype: int64

Number of Missing Intervals:
332

Number of Intervals Greater Than 10 Seconds:
50816

Number of Non-Positive Intervals:
0

Total Rows: 1516948
Total Segments: 332


In [18]:
WINDOW_SECONDS = 600

window_checks = []

for segment_id, group in data.groupby("segment", sort=True):
    group = group.sort_values("timestamp").reset_index(drop=True)

    window_start = group["timestamp"].iloc[0]
    segment_end = group["timestamp"].iloc[-1]

    while window_start + pd.Timedelta(
        seconds=WINDOW_SECONDS
    ) <= segment_end:
        window_end = window_start + pd.Timedelta(
            seconds=WINDOW_SECONDS
        )

        window_data = group[
            (group["timestamp"] >= window_start) &
            (group["timestamp"] < window_end)
        ]

        intervals = window_data["timestamp"].diff().dt.total_seconds()

        window_checks.append({
            "segment": segment_id,
            "window_start": window_start,
            "window_end": window_end,
            "n_rows": len(window_data),
            "has_gap": (intervals > 10).any()
        })

        window_start = window_end

window_checks_df = pd.DataFrame(window_checks)

print("Total complete-duration windows:", len(window_checks_df))
print("\nRows per window:")
print(window_checks_df["n_rows"].describe())

print("\nWindows containing gaps:", window_checks_df["has_gap"].sum())
print("Windows with exactly 60 rows:",
      (window_checks_df["n_rows"] == 60).sum())

Total complete-duration windows: 25055

Rows per window:
count    25055.000000
mean        60.094073
std          2.191856
min         49.000000
25%         60.000000
50%         61.000000
75%         61.000000
max         61.000000
Name: n_rows, dtype: float64

Windows containing gaps: 1103
Windows with exactly 60 rows: 11190


In [19]:
print("Missing values per column:")
print(df.isnull().sum())

print("\nTotal missing values:")
print(df.isnull().sum().sum())

print("\nDuplicate rows:")
print(df.duplicated().sum())

print("\nDuplicate timestamps within segments:")
print(
    df.duplicated(subset=["segment", "timestamp"]).sum()
)

Missing values per column:
timestamp          0
TP2                0
TP3                0
H1                 0
DV_pressure        0
Reservoirs         0
Oil_temperature    0
Motor_current      0
COMP               0
DV_eletric         0
Towers             0
MPG                0
LPS                0
Pressure_switch    0
Oil_level          0
Caudal_impulses    0
segment            0
ground_truth       0
dtype: int64

Total missing values:
0

Duplicate rows:
0

Duplicate timestamps within segments:
0


In [20]:
import numpy as np

sensor_columns = [
    "TP2", "TP3", "H1", "DV_pressure",
    "Reservoirs", "Oil_temperature", "Motor_current",
    "COMP", "DV_eletric", "Towers", "MPG", "LPS",
    "Pressure_switch", "Oil_level", "Caudal_impulses"
]

print("Sensor data types:")
print(df[sensor_columns].dtypes)

print("\nNon-numeric values:")
print(
    df[sensor_columns]
    .apply(pd.to_numeric, errors="coerce")
    .isna()
    .sum()
)

print("\nInfinite values:")
print(
    np.isinf(
        df[sensor_columns]
        .select_dtypes(include="number")
    ).sum()
)

Sensor data types:
TP2                float64
TP3                float64
H1                 float64
DV_pressure        float64
Reservoirs         float64
Oil_temperature    float64
Motor_current      float64
COMP               float64
DV_eletric         float64
Towers             float64
MPG                float64
LPS                float64
Pressure_switch    float64
Oil_level          float64
Caudal_impulses    float64
dtype: object

Non-numeric values:
TP2                0
TP3                0
H1                 0
DV_pressure        0
Reservoirs         0
Oil_temperature    0
Motor_current      0
COMP               0
DV_eletric         0
Towers             0
MPG                0
LPS                0
Pressure_switch    0
Oil_level          0
Caudal_impulses    0
dtype: int64

Infinite values:
TP2                0
TP3                0
H1                 0
DV_pressure        0
Reservoirs         0
Oil_temperature    0
Motor_current      0
COMP               0
DV_eletric         0
Towers

In [21]:
import pandas as pd

WINDOW_SECONDS = 600

data = df.copy()
data["timestamp"] = pd.to_datetime(data["timestamp"])

data = data.sort_values(
    ["segment", "timestamp"]
).reset_index(drop=True)

windows = []

for segment_id, group in data.groupby("segment", sort=True):
    group = group.sort_values("timestamp").reset_index(drop=True)

    window_start = group["timestamp"].iloc[0]
    segment_end = group["timestamp"].iloc[-1]

    while window_start + pd.Timedelta(
        seconds=WINDOW_SECONDS
    ) <= segment_end:

        window_end = window_start + pd.Timedelta(
            seconds=WINDOW_SECONDS
        )

        window_data = group[
            (group["timestamp"] >= window_start) &
            (group["timestamp"] < window_end)
        ]

        if not window_data.empty:
            windows.append({
                "segment": segment_id,
                "window_start": window_start,
                "window_end": window_end,
                "n_rows": len(window_data),
                "n_abnormal": int(
                    window_data["ground_truth"].sum()
                )
            })

        window_start = window_end

windows_df = pd.DataFrame(windows)

print("Total windows:", len(windows_df))
print("\nFirst 10 windows:")
print(windows_df.head(10).to_string(index=False))

Total windows: 25055

First 10 windows:
 segment        window_start          window_end  n_rows  n_abnormal
       0 2020-02-01 00:00:00 2020-02-01 00:10:00      61           0
       0 2020-02-01 00:10:00 2020-02-01 00:20:00      61           0
       0 2020-02-01 00:20:00 2020-02-01 00:30:00      60           0
       0 2020-02-01 00:30:00 2020-02-01 00:40:00      61           0
       0 2020-02-01 00:40:00 2020-02-01 00:50:00      60           0
       0 2020-02-01 00:50:00 2020-02-01 01:00:00      61           0
       0 2020-02-01 01:00:00 2020-02-01 01:10:00      60           0
       0 2020-02-01 01:10:00 2020-02-01 01:20:00      61           0
       0 2020-02-01 01:20:00 2020-02-01 01:30:00      60           0
       0 2020-02-01 01:30:00 2020-02-01 01:40:00      61           0


In [22]:
print("Total windows:", len(windows_df))

print("\nRows per window:")
print(windows_df["n_rows"].describe())

print("\nWindows with fewer than 50 rows:")
print((windows_df["n_rows"] < 50).sum())

print("\nWindows with abnormal readings:")
print((windows_df["n_abnormal"] > 0).sum())

print("\nTotal abnormal readings inside windows:")
print(windows_df["n_abnormal"].sum())

print("\nWindows by segment:")
print(windows_df.groupby("segment").size().describe())

Total windows: 25055

Rows per window:
count    25055.000000
mean        60.094073
std          2.191856
min         49.000000
25%         60.000000
50%         61.000000
75%         61.000000
max         61.000000
Name: n_rows, dtype: float64

Windows with fewer than 50 rows:
294

Windows with abnormal readings:
57

Total abnormal readings inside windows:
3385

Windows by segment:
count    275.000000
mean      91.109091
std       98.409518
min        1.000000
25%        7.000000
50%       76.000000
75%      124.500000
max      604.000000
dtype: float64


In [23]:
short_windows = windows_df[
    windows_df["n_rows"] < 50
].copy()

print("Number of short windows:", len(short_windows))

print("\nFirst 20 short windows:")
print(short_windows.head(20).to_string(index=False))

print("\nShort windows by segment:")
print(
    short_windows.groupby("segment")
    .size()
    .sort_values(ascending=False)
    .head(20)
)

Number of short windows: 294

First 20 short windows:
 segment        window_start          window_end  n_rows  n_abnormal
     104 2020-04-13 18:59:08 2020-04-13 19:09:08      49           0
     112 2020-04-17 09:58:50 2020-04-17 10:08:50      49           0
     112 2020-04-17 10:38:50 2020-04-17 10:48:50      49           0
     112 2020-04-17 11:08:50 2020-04-17 11:18:50      49           0
     112 2020-04-17 11:48:50 2020-04-17 11:58:50      49           0
     112 2020-04-17 12:28:50 2020-04-17 12:38:50      49           0
     112 2020-04-17 12:58:50 2020-04-17 13:08:50      49           0
     112 2020-04-17 13:38:50 2020-04-17 13:48:50      49           0
     112 2020-04-17 14:18:50 2020-04-17 14:28:50      49           0
     112 2020-04-17 14:58:50 2020-04-17 15:08:50      49           0
     112 2020-04-17 15:28:50 2020-04-17 15:38:50      49           0
     112 2020-04-17 15:58:50 2020-04-17 16:08:50      49           0
     112 2020-04-17 16:38:50 2020-04-17 16:48:50 

In [24]:
segments_to_check = [214, 175, 115, 176, 200, 112]

for segment_id in segments_to_check:
    group = (
        data[data["segment"] == segment_id]
        .sort_values("timestamp")
        .copy()
    )

    group["time_diff"] = (
        group["timestamp"]
        .diff()
        .dt.total_seconds()
    )

    print(f"\nSegment {segment_id}")
    print("Total rows:", len(group))
    print("Median interval:", group["time_diff"].median())
    print("Intervals greater than 10 seconds:",
          (group["time_diff"] > 10).sum())
    print("Intervals greater than 30 seconds:",
          (group["time_diff"] > 30).sum())
    print("Maximum interval:", group["time_diff"].max())


Segment 214
Total rows: 21061
Median interval: 12.0
Intervals greater than 10 seconds: 18490
Intervals greater than 30 seconds: 0
Maximum interval: 23.0

Segment 175
Total rows: 9667
Median interval: 12.0
Intervals greater than 10 seconds: 8456
Intervals greater than 30 seconds: 0
Maximum interval: 13.0

Segment 115
Total rows: 15627
Median interval: 10.0
Intervals greater than 10 seconds: 6090
Intervals greater than 30 seconds: 0
Maximum interval: 14.0

Segment 176
Total rows: 3983
Median interval: 12.0
Intervals greater than 10 seconds: 3982
Intervals greater than 30 seconds: 0
Maximum interval: 13.0

Segment 200
Total rows: 4906
Median interval: 12.0
Intervals greater than 10 seconds: 4480
Intervals greater than 30 seconds: 0
Maximum interval: 22.0

Segment 112
Total rows: 6175
Median interval: 12.0
Intervals greater than 10 seconds: 4442
Intervals greater than 30 seconds: 0
Maximum interval: 13.0


In [25]:
print("Original abnormal readings:", int(df["ground_truth"].sum()))

print(
    "Abnormal readings inside windows:",
    int(windows_df["n_abnormal"].sum())
)

print(
    "Difference:",
    int(df["ground_truth"].sum() - windows_df["n_abnormal"].sum())
)

print("\nSegments containing abnormal labels:")
print(
    df.groupby("segment")["ground_truth"]
    .sum()
    .loc[lambda values: values > 0]
)

Original abnormal readings: 3415
Abnormal readings inside windows: 3385
Difference: 30

Segments containing abnormal labels:
segment
102     909
123      30
124    2476
Name: ground_truth, dtype: int64


In [26]:
segment_123 = (
    data[data["segment"] == 123]
    .sort_values("timestamp")
    .copy()
)

print("Total rows:", len(segment_123))
print("First timestamp:", segment_123["timestamp"].min())
print("Last timestamp:", segment_123["timestamp"].max())
print("Abnormal readings:", int(segment_123["ground_truth"].sum()))

print("\nSegment 123 timestamps:")
print(segment_123[["timestamp", "ground_truth"]].to_string(index=False))

print("\nWindows for segment 123:")
print(
    windows_df[windows_df["segment"] == 123]
    .to_string(index=False)
)

Total rows: 30
First timestamp: 2020-04-27 01:12:49
Last timestamp: 2020-04-27 01:17:36
Abnormal readings: 30

Segment 123 timestamps:
          timestamp  ground_truth
2020-04-27 01:12:49             1
2020-04-27 01:12:59             1
2020-04-27 01:13:09             1
2020-04-27 01:13:19             1
2020-04-27 01:13:29             1
2020-04-27 01:13:38             1
2020-04-27 01:13:48             1
2020-04-27 01:13:58             1
2020-04-27 01:14:08             1
2020-04-27 01:14:18             1
2020-04-27 01:14:28             1
2020-04-27 01:14:38             1
2020-04-27 01:14:48             1
2020-04-27 01:14:58             1
2020-04-27 01:15:08             1
2020-04-27 01:15:18             1
2020-04-27 01:15:27             1
2020-04-27 01:15:37             1
2020-04-27 01:15:47             1
2020-04-27 01:15:57             1
2020-04-27 01:16:07             1
2020-04-27 01:16:17             1
2020-04-27 01:16:27             1
2020-04-27 01:16:37             1
2020-04-27 01:1

In [27]:
print("Total windows:", len(windows_df))

print("\nWindow label distribution:")
print(
    (windows_df["n_abnormal"] > 0)
    .value_counts()
    .rename(index={False: "Normal", True: "Abnormal"})
)

print("\nAbnormal readings per window:")
print(
    windows_df["n_abnormal"]
    .describe()
)

Total windows: 25055

Window label distribution:
n_abnormal
Normal      24998
Abnormal       57
Name: count, dtype: int64

Abnormal readings per window:
count    25055.000000
mean         0.135103
std          2.842010
min          0.000000
25%          0.000000
50%          0.000000
75%          0.000000
max         61.000000
Name: n_abnormal, dtype: float64


In [28]:
windows_df["window_label"] = (
    windows_df["n_abnormal"] > 0
).astype(int)

print("Total windows:", len(windows_df))

print("\nWindow label distribution:")
print(windows_df["window_label"].value_counts().sort_index())

print("\nLabel meaning:")
print("0 = Normal")
print("1 = Abnormal")

print("\nWindows with abnormal readings:")
print(
    windows_df.loc[
        windows_df["window_label"] == 1,
        ["segment", "window_start", "window_end",
         "n_rows", "n_abnormal", "window_label"]
    ].head(10).to_string(index=False)
)

Total windows: 25055

Window label distribution:
window_label
0    24998
1       57
Name: count, dtype: int64

Label meaning:
0 = Normal
1 = Abnormal

Windows with abnormal readings:
 segment        window_start          window_end  n_rows  n_abnormal  window_label
     102 2020-04-12 09:54:35 2020-04-12 10:04:35      60          28             1
     102 2020-04-12 10:04:35 2020-04-12 10:14:35      61          61             1
     102 2020-04-12 10:14:35 2020-04-12 10:24:35      60          60             1
     102 2020-04-12 10:24:35 2020-04-12 10:34:35      61          61             1
     102 2020-04-12 10:34:35 2020-04-12 10:44:35      60          60             1
     102 2020-04-12 10:44:35 2020-04-12 10:54:35      61          61             1
     102 2020-04-12 10:54:35 2020-04-12 11:04:35      60          60             1
     102 2020-04-12 11:04:35 2020-04-12 11:14:35      61          61             1
     102 2020-04-12 11:14:35 2020-04-12 11:24:35      60          60  

In [29]:
sensor_columns = [
    "TP2", "TP3", "H1", "DV_pressure",
    "Reservoirs", "Oil_temperature", "Motor_current",
    "COMP", "DV_eletric", "Towers", "MPG", "LPS",
    "Pressure_switch", "Oil_level", "Caudal_impulses"
]

feature_rows = []

for _, window in windows_df.iterrows():
    group = data[
        (data["segment"] == window["segment"]) &
        (data["timestamp"] >= window["window_start"]) &
        (data["timestamp"] < window["window_end"])
    ]

    row = {
        "segment": window["segment"],
        "window_start": window["window_start"],
        "window_end": window["window_end"],
        "window_label": window["window_label"]
    }

    for column in sensor_columns:
        row[f"{column}_mean"] = group[column].mean()
        row[f"{column}_std"] = group[column].std()
        row[f"{column}_min"] = group[column].min()
        row[f"{column}_max"] = group[column].max()

    feature_rows.append(row)

features_df = pd.DataFrame(feature_rows)

print("Feature table shape:", features_df.shape)
print("\nFirst 5 rows:")
print(features_df.head().to_string(index=False))

print("\nMissing values:", features_df.isnull().sum().sum())

Feature table shape: (25055, 64)

First 5 rows:
 segment        window_start          window_end  window_label  TP2_mean  TP2_std  TP2_min  TP2_max  TP3_mean  TP3_std  TP3_min  TP3_max  H1_mean   H1_std  H1_min  H1_max  DV_pressure_mean  DV_pressure_std  DV_pressure_min  DV_pressure_max  Reservoirs_mean  Reservoirs_std  Reservoirs_min  Reservoirs_max  Oil_temperature_mean  Oil_temperature_std  Oil_temperature_min  Oil_temperature_max  Motor_current_mean  Motor_current_std  Motor_current_min  Motor_current_max  COMP_mean  COMP_std  COMP_min  COMP_max  DV_eletric_mean  DV_eletric_std  DV_eletric_min  DV_eletric_max  Towers_mean  Towers_std  Towers_min  Towers_max  MPG_mean  MPG_std  MPG_min  MPG_max  LPS_mean  LPS_std  LPS_min  LPS_max  Pressure_switch_mean  Pressure_switch_std  Pressure_switch_min  Pressure_switch_max  Oil_level_mean  Oil_level_std  Oil_level_min  Oil_level_max  Caudal_impulses_mean  Caudal_impulses_std  Caudal_impulses_min  Caudal_impulses_max
       0 2020-02-01 00:00

In [30]:
feature_columns = [
    column for column in features_df.columns
    if column not in [
        "segment",
        "window_start",
        "window_end",
        "window_label"
    ]
]

print("Number of feature columns:", len(feature_columns))

print("\nFeatures with zero variance:")
print(
    features_df[feature_columns]
    .std()
    .loc[lambda values: values == 0]
)

print("\nFeatures with missing values:")
print(
    features_df[feature_columns]
    .isnull()
    .sum()
    .loc[lambda values: values > 0]
)

print("\nLabel distribution:")
print(features_df["window_label"].value_counts())

Number of feature columns: 60

Features with zero variance:
Towers_max             0.0
Pressure_switch_max    0.0
dtype: float64

Features with missing values:
Series([], dtype: int64)

Label distribution:
window_label
0    24998
1       57
Name: count, dtype: int64


In [31]:

feature_columns = [
    column for column in features_df.columns
    if column not in [
        "segment",
        "window_start",
        "window_end",
        "window_label"
    ]
]



constant_features = [
    "Towers_max",
    "Pressure_switch_max"
]

feature_columns = [
    column for column in feature_columns
    if column not in constant_features
]


X = features_df[feature_columns].copy()
y = features_df["window_label"].copy()


print("Features shape:", X.shape)
print("Labels shape:", y.shape)
print("Number of features:", len(feature_columns))
print("Missing values:", X.isnull().sum().sum())

print("\nLabel distribution:")
print(y.value_counts())

Features shape: (25055, 58)
Labels shape: (25055,)
Number of features: 58
Missing values: 0

Label distribution:
window_label
0    24998
1       57
Name: count, dtype: int64


In [32]:
import pandas as pd
import numpy as np

data_with_windows = df.copy()
data_with_windows["timestamp"] = pd.to_datetime(
    data_with_windows["timestamp"]
)

windows_export = windows_df.copy()

windows_export = windows_export.sort_values(
    ["segment", "window_start"]
).reset_index(drop=True)

windows_export["window_index"] = (
    windows_export.groupby("segment").cumcount()
)

windows_export["window_id"] = (
    "segment_"
    + windows_export["segment"].astype(str)
    + "_window_"
    + (windows_export["window_index"] + 1).astype(str)
)

segment_starts = data_with_windows.groupby(
    "segment"
)["timestamp"].min()

data_with_windows["segment_start"] = (
    data_with_windows["segment"].map(segment_starts)
)

data_with_windows["window_index"] = (
    (
        data_with_windows["timestamp"]
        - data_with_windows["segment_start"]
    ).dt.total_seconds() // 600
).astype(int)

window_id_map = windows_export.set_index(
    ["segment", "window_index"]
)["window_id"]

row_keys = pd.MultiIndex.from_arrays(
    [
        data_with_windows["segment"],
        data_with_windows["window_index"]
    ],
    names=["segment", "window_index"]
)

data_with_windows["window_id"] = (
    window_id_map.reindex(row_keys).to_numpy()
)

data_with_windows.drop(
    columns=["segment_start", "window_index"],
    inplace=True
)

print("Original rows:", len(df))
print("Final rows:", len(data_with_windows))
print(
    "Rows assigned to windows:",
    data_with_windows["window_id"].notna().sum()
)
print(
    "Rows without a complete window:",
    data_with_windows["window_id"].isna().sum()
)
print(
    "Abnormal readings without a complete window:",
    data_with_windows.loc[
        data_with_windows["window_id"].isna(),
        "ground_truth"
    ].sum()
)

Original rows: 1516948
Final rows: 1516948
Rows assigned to windows: 1505657
Rows without a complete window: 11291
Abnormal readings without a complete window: 30


In [33]:
import pandas as pd

unassigned = data_with_windows[
    data_with_windows["window_id"].isna()
].copy()

unassigned = unassigned.sort_values(
    ["segment", "timestamp"]
)

incomplete_parts = []

for segment_id, group in unassigned.groupby("segment"):
    group = group.sort_values("timestamp").copy()

    group["time_gap"] = (
        group["timestamp"].diff().dt.total_seconds()
    )

    group["new_part"] = (
        group["time_gap"].isna()
        | (group["time_gap"] > 30)
    )

    group["part_number"] = group["new_part"].cumsum()

    for part_number, part in group.groupby("part_number"):
        incomplete_parts.append({
            "segment": segment_id,
            "part_number": int(part_number),
            "start_time": part["timestamp"].min(),
            "end_time": part["timestamp"].max(),
            "duration_seconds": (
                part["timestamp"].max()
                - part["timestamp"].min()
            ).total_seconds(),
            "n_rows": len(part),
            "n_abnormal": int(part["ground_truth"].sum())
        })

incomplete_parts_df = pd.DataFrame(incomplete_parts)

print("Number of incomplete parts:", len(incomplete_parts_df))
print("Total unassigned readings:", len(unassigned))
print(
    "Abnormal readings in incomplete parts:",
    incomplete_parts_df["n_abnormal"].sum()
)

display(incomplete_parts_df.head(20))

Number of incomplete parts: 332
Total unassigned readings: 11291
Abnormal readings in incomplete parts: 30


,segment,part_number,start_time,end_time,duration_seconds,n_rows,n_abnormal
0,0,1,2020-02-01 12:40:05,2020-02-01 12:48:40,515.0,53,0
1,1,1,2020-02-01 19:33:47,2020-02-01 19:40:04,377.0,39,0
2,2,1,2020-02-01 23:15:33,2020-02-01 23:20:21,288.0,30,0
3,3,1,2020-02-02 07:34:32,2020-02-02 07:37:40,188.0,20,0
4,4,1,2020-02-05 00:31:52,2020-02-05 00:32:21,29.0,4,0
5,5,1,2020-02-05 01:03:00,2020-02-05 01:07:37,277.0,29,0
6,6,1,2020-02-06 01:58:46,2020-02-06 02:07:01,495.0,51,0
7,7,1,2020-02-06 14:02:26,2020-02-06 14:08:53,387.0,40,0
8,8,1,2020-02-06 19:09:28,2020-02-06 19:16:35,427.0,44,0
9,9,1,2020-02-06 23:43:24,2020-02-06 23:48:02,278.0,29,0


In [34]:
print("Incomplete parts:", len(incomplete_parts_df))

print("\nDuration statistics (seconds):")
print(incomplete_parts_df["duration_seconds"].describe())

print("\nShortest 10 parts:")
display(
    incomplete_parts_df.sort_values(
        "duration_seconds"
    ).head(10)
)

print("\nLongest 10 parts:")
display(
    incomplete_parts_df.sort_values(
        "duration_seconds",
        ascending=False
    ).head(10)
)

print("\nParts containing abnormal readings:")
display(
    incomplete_parts_df[
        incomplete_parts_df["n_abnormal"] > 0
    ]
)

Incomplete parts: 332

Duration statistics (seconds):
count    332.000000
mean     328.472892
std      162.626151
min        0.000000
25%      238.000000
50%      287.000000
75%      495.000000
max      595.000000
Name: duration_seconds, dtype: float64

Shortest 10 parts:


,segment,part_number,start_time,end_time,duration_seconds,n_rows,n_abnormal
303,303,1,2020-08-15 19:30:42,2020-08-15 19:30:42,0.0,1,0
68,68,1,2020-03-19 21:06:46,2020-03-19 21:06:46,0.0,1,0
115,115,1,2020-04-21 01:16:52,2020-04-21 01:16:52,0.0,1,0
256,256,1,2020-07-17 00:36:52,2020-07-17 00:37:01,9.0,2,0
130,130,1,2020-04-30 10:35:14,2020-04-30 10:35:24,10.0,2,0
181,181,1,2020-05-30 07:52:34,2020-05-30 07:52:44,10.0,2,0
36,36,1,2020-03-02 20:50:18,2020-03-02 20:50:37,19.0,3,0
124,124,1,2020-04-27 19:32:04,2020-04-27 19:32:23,19.0,3,0
211,211,1,2020-06-21 23:48:00,2020-06-21 23:48:20,20.0,3,0
4,4,1,2020-02-05 00:31:52,2020-02-05 00:32:21,29.0,4,0



Longest 10 parts:


,segment,part_number,start_time,end_time,duration_seconds,n_rows,n_abnormal
316,316,1,2020-08-22 19:01:49,2020-08-22 19:11:44,595.0,61,0
177,177,1,2020-05-28 03:22:24,2020-05-28 03:32:10,586.0,60,0
228,228,1,2020-07-03 03:14:43,2020-07-03 03:24:29,586.0,60,0
19,19,1,2020-02-12 00:59:50,2020-02-12 01:09:35,585.0,60,0
273,273,1,2020-07-26 01:17:39,2020-07-26 01:27:24,585.0,60,0
28,28,1,2020-02-20 01:24:21,2020-02-20 01:34:06,585.0,60,0
88,88,1,2020-04-07 10:25:32,2020-04-07 10:35:17,585.0,60,0
63,63,1,2020-03-15 00:59:15,2020-03-15 01:09:00,585.0,60,0
223,223,1,2020-07-03 01:32:33,2020-07-03 01:42:18,585.0,60,0
13,13,1,2020-02-09 00:40:11,2020-02-09 00:49:56,585.0,60,0



Parts containing abnormal readings:


,segment,part_number,start_time,end_time,duration_seconds,n_rows,n_abnormal
123,123,1,2020-04-27 01:12:49,2020-04-27 01:17:36,287.0,30,30


In [35]:
incomplete_parts_df.to_csv(
    "incomplete_parts_summary.csv",
    index=False
)

print("Incomplete parts report saved.")
print("Number of parts:", len(incomplete_parts_df))
print(
    "Parts shorter than 10 minutes:",
    (
        incomplete_parts_df["duration_seconds"] < 600
    ).sum()
)
print(
    "Parts containing anomalies:",
    (
        incomplete_parts_df["n_abnormal"] > 0
    ).sum()
)

Incomplete parts report saved.
Number of parts: 332
Parts shorter than 10 minutes: 332
Parts containing anomalies: 1


In [36]:
import pandas as pd
import numpy as np
import json

window_size_seconds = 600
stride_seconds = 600
gap_threshold_seconds = 30

source = df.copy()
source["_original_order"] = np.arange(len(source))
source["timestamp"] = pd.to_datetime(source["timestamp"], errors="raise")
source = source.sort_values(["segment", "timestamp"]).reset_index(drop=True)

source["_segment_start"] = source.groupby("segment")["timestamp"].transform("min")
source["_elapsed_seconds"] = (
    source["timestamp"] - source["_segment_start"]
).dt.total_seconds()

source["_time_bin"] = (
    source["_elapsed_seconds"] // window_size_seconds
).astype(int)

source["_gap_seconds"] = (
    source.groupby("segment")["timestamp"]
    .diff()
    .dt.total_seconds()
)

source["_new_window"] = (
    source["_gap_seconds"].isna()
    | source["_time_bin"].ne(
        source.groupby("segment")["_time_bin"].shift()
    )
    | source["_gap_seconds"].gt(gap_threshold_seconds)
)

source["_window_number"] = (
    source.groupby("segment")["_new_window"].cumsum().astype(int)
)

source["window_id"] = (
    "segment_"
    + source["segment"].astype(str)
    + "_window_"
    + source["_window_number"].astype(str)
)

windows_index = (
    source.groupby(["segment", "window_id"], as_index=False)
    .agg(
        start_time=("timestamp", "min"),
        end_time=("timestamp", "max"),
        n_rows=("timestamp", "size"),
        n_abnormal=("ground_truth", "sum")
    )
)

windows_index["window_label"] = (
    windows_index["n_abnormal"] > 0
).astype(int)

windows_index = windows_index[
    [
        "window_id",
        "segment",
        "start_time",
        "end_time",
        "n_rows",
        "window_label",
        "n_abnormal"
    ]
]

data_windowed = (
    source.sort_values("_original_order")
    .drop(
        columns=[
            "_original_order",
            "_segment_start",
            "_elapsed_seconds",
            "_time_bin",
            "_gap_seconds",
            "_new_window",
            "_window_number"
        ]
    )
    .reset_index(drop=True)
)

gap_check = source.groupby("window_id")["_gap_seconds"].max()
duration_check = (
    windows_index["end_time"] - windows_index["start_time"]
).dt.total_seconds()

config = {
    "window_size": "10 minutes",
    "window_size_seconds": window_size_seconds,
    "stride": "10 minutes",
    "stride_seconds": stride_seconds,
    "window_alignment": "segment_start",
    "overlap": False,
    "gap_threshold_seconds": gap_threshold_seconds,
    "gap_policy": "Start a new window when consecutive readings are more than 30 seconds apart.",
    "allow_partial_windows": True,
    "labeling_rule": "window_label = 1 if any reading in the window has ground_truth = 1; otherwise 0.",
    "note": "The 30-second gap threshold is a provisional operational choice and should be confirmed by the team."
}

data_windowed.to_csv("data_windowed.csv", index=False)
windows_index.to_csv("windows_index.csv", index=False)

with open("windowing_config.json", "w", encoding="utf-8") as file:
    json.dump(config, file, indent=2, ensure_ascii=False)

print("Original rows:", len(df))
print("Final rows:", len(data_windowed))
print("Rows without window_id:", data_windowed["window_id"].isna().sum())
print("Number of windows:", len(windows_index))
print("Original anomalies:", int(df["ground_truth"].sum()))
print("Final anomalies:", int(data_windowed["ground_truth"].sum()))
print("Windows containing anomalies:", int(windows_index["window_label"].sum()))
print("Maximum gap within a window (seconds):", gap_check.max())
print("Maximum window duration (seconds):", duration_check.max())
print("Files saved: data_windowed.csv, windows_index.csv, windowing_config.json")

Original rows: 1516948
Final rows: 1516948
Rows without window_id: 0
Number of windows: 25387
Original anomalies: 3415
Final anomalies: 3415
Windows containing anomalies: 58
Maximum gap within a window (seconds): 30.0
Maximum window duration (seconds): 598.0
Files saved: data_windowed.csv, windows_index.csv, windowing_config.json


In [37]:
check = data_windowed.sort_values(
    ["segment", "timestamp"]
).copy()

check["gap_in_window"] = (
    check.groupby("window_id")["timestamp"]
    .diff()
    .dt.total_seconds()
)

window_check = check.groupby("window_id").agg(
    segment_count=("segment", "nunique"),
    max_gap_seconds=("gap_in_window", "max"),
    n_rows=("timestamp", "size")
)

print("Windows crossing segments:", (window_check["segment_count"] > 1).sum())
print("Windows with gaps over 30 seconds:", (window_check["max_gap_seconds"] > 30).sum())
print("Windows with more than one segment:", (window_check["segment_count"] > 1).sum())
print("Total windows checked:", len(window_check))
print("Files to submit: data_windowed.csv, windows_index.csv, windowing_config.json")

Windows crossing segments: 0
Windows with gaps over 30 seconds: 0
Windows with more than one segment: 0
Total windows checked: 25387
Files to submit: data_windowed.csv, windows_index.csv, windowing_config.json


In [38]:
index_check = windows_index.set_index("window_id").sort_index()

counts_check = (
    data_windowed.groupby("window_id")
    .agg(
        actual_rows=("timestamp", "size"),
        actual_start=("timestamp", "min"),
        actual_end=("timestamp", "max"),
        actual_label=("ground_truth", lambda x: int(x.max() > 0))
    )
    .sort_index()
)

print("Same window IDs:", index_check.index.equals(counts_check.index))
print("Same row counts:", index_check["n_rows"].equals(counts_check["actual_rows"]))
print("Same start times:", (
    pd.to_datetime(index_check["start_time"]).reset_index(drop=True)
    == pd.to_datetime(counts_check["actual_start"]).reset_index(drop=True)
).all())
print("Same end times:", (
    pd.to_datetime(index_check["end_time"]).reset_index(drop=True)
    == pd.to_datetime(counts_check["actual_end"]).reset_index(drop=True)
).all())
print("Same labels:", (
    index_check["window_label"].reset_index(drop=True)
    == counts_check["actual_label"].reset_index(drop=True)
).all())

Same window IDs: True
Same row counts: True
Same start times: True
Same end times: True
Same labels: True
